# OCR Model Comparison: RapidOCR vs. TrOCR

A small, reproducible experiment for [personal-research issue #80](https://github.com/pomodorozhong/personal-research/issues/80). We compare a compact dedicated OCR pipeline with a Transformer-based recognizer on the same controlled images.

- **RapidOCR** is the dedicated OCR baseline. It runs compact PP-OCR models through ONNX Runtime, and its Python wheel includes the default model files.
- **Microsoft TrOCR Small Printed** is the LLM-adjacent model. It combines an image Transformer encoder with an autoregressive text Transformer decoder. It is a specialized OCR model, not a general-purpose LLM.


## 1. Question and experimental design

**Question:** How do a compact OCR-specific recognizer and Transformer text generation differ in accuracy and latency as simple printed text is degraded?

The notebook generates single-line English images under five conditions: clean, low contrast, blur, rotation, and Gaussian noise. Each recognizer receives the exact same image. We report:

- **Character error rate (CER):** character edits divided by reference characters; lower is better.
- **Word error rate (WER):** word edits divided by reference words; lower is better.
- **Warm inference latency:** model/engine loading and the first warm-up call are excluded. Both engines stay loaded in the notebook process; different batch sizes and runtimes would change the comparison.

> **Scope:** TrOCR's published checkpoint is intended for single text-line recognition, not full-page layout or text detection. Using pre-cropped lines keeps the comparison fair. This tiny synthetic corpus is a learning exercise, not a production benchmark.


## 2. Imports and reproducibility

Run this notebook from `researches/ocr` after `uv sync --locked`. RapidOCR, its bundled PP-OCR models, ONNX Runtime, PyTorch, and Transformers are all installed from the locked `uv` environment. TrOCR checkpoint data is downloaded to the Hugging Face cache on first use.


In [ ]:
from __future__ import annotations

import platform
import re
import time
from dataclasses import dataclass

import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont
from rapidocr import RapidOCR

SEED = 80
RNG = np.random.default_rng(SEED)
MODEL_ID = "microsoft/trocr-small-printed"
RUN_RAPIDOCR = True
RUN_TROCR = True

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")


## 3. Build a controlled OCR corpus

The generator keeps the ground truth known and changes one nuisance variable at a time. The samples mix letters, digits, punctuation, and currency.


In [ ]:
TEXTS = [
    "Invoice 2026-09",
    "Total: $42.70",
    "Room B-204",
    "Open source OCR",
    "Code: A1B2C3",
]
CONDITIONS = ["clean", "low_contrast", "blur", "rotation", "noise"]


def load_font(size: int = 46) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/Library/Fonts/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size=size)
        except OSError:
            pass
    return ImageFont.load_default(size=size)


FONT = load_font()


def render_line(text: str, condition: str, seed: int) -> Image.Image:
    background = 250
    image = Image.new("L", (760, 100), color=background)
    draw = ImageDraw.Draw(image)
    bbox = draw.textbbox((0, 0), text, font=FONT)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    x = max(24, (image.width - text_w) // 2)
    y = (image.height - text_h) // 2 - bbox[1]
    ink = 18 if condition != "low_contrast" else 145
    draw.text((x, y), text, fill=ink, font=FONT)

    if condition == "blur":
        image = image.filter(ImageFilter.GaussianBlur(radius=1.8))
    elif condition == "rotation":
        image = image.rotate(2.8, resample=Image.Resampling.BICUBIC, fillcolor=background)
    elif condition == "noise":
        local_rng = np.random.default_rng(seed)
        pixels = np.asarray(image, dtype=np.float32)
        pixels += local_rng.normal(loc=0.0, scale=24.0, size=pixels.shape)
        image = Image.fromarray(np.clip(pixels, 0, 255).astype(np.uint8), mode="L")

    return image.convert("RGB")


@dataclass(frozen=True)
class Sample:
    sample_id: str
    condition: str
    truth: str
    image: Image.Image


samples = [
    Sample(
        sample_id=f"{condition}-{text_index}",
        condition=condition,
        truth=text,
        image=render_line(text, condition, SEED + condition_index * 10 + text_index),
    )
    for condition_index, condition in enumerate(CONDITIONS)
    for text_index, text in enumerate(TEXTS)
]

print(f"Created {len(samples)} samples ({len(TEXTS)} texts × {len(CONDITIONS)} conditions).")


In [ ]:
fig, axes = plt.subplots(len(CONDITIONS), 1, figsize=(11, 7.5))
for ax, condition in zip(axes, CONDITIONS):
    sample = next(item for item in samples if item.condition == condition)
    ax.imshow(sample.image)
    ax.set_title(f"{condition}: {sample.truth!r}", loc="left")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Shared evaluation helpers

Whitespace is normalized because OCR engines often add trailing newlines. We retain **strict** case-sensitive CER/WER for fidelity, and also report **case-folded** CER/WER so a checkpoint's casing convention does not hide its transcription quality. CER or WER can exceed 1.0 when a prediction contains many insertions.


In [ ]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def score_prediction(model: str, sample: Sample, prediction: str, latency_s: float) -> dict:
    truth = normalize_text(sample.truth)
    prediction = normalize_text(prediction)
    truth_folded = truth.casefold()
    prediction_folded = prediction.casefold()
    return {
        "model": model,
        "sample_id": sample.sample_id,
        "condition": sample.condition,
        "truth": truth,
        "prediction": prediction,
        "exact_match": truth == prediction,
        "cer": jiwer.cer(truth, prediction),
        "wer": jiwer.wer(truth, prediction),
        "cer_casefold": jiwer.cer(truth_folded, prediction_folded),
        "wer_casefold": jiwer.wer(truth_folded, prediction_folded),
        "latency_ms": latency_s * 1_000,
    }


## 5. Dedicated OCR baseline: RapidOCR

RapidOCR is a complete OCR pipeline with detection, orientation classification, and recognition. Because the samples are already line crops, we disable detection and classification and run only its PP-OCR recognizer through ONNX Runtime. This matches TrOCR's line-recognition scope and keeps the model loaded between calls.


In [ ]:
rapidocr_rows: list[dict] = []
rapidocr_engine = None


def rapidocr_predict(image: Image.Image) -> str:
    assert rapidocr_engine is not None
    result = rapidocr_engine(image, use_det=False, use_cls=False, use_rec=True)
    return result.txts[0] if result.txts else ""


if RUN_RAPIDOCR:
    rapidocr_engine = RapidOCR(params={
        "Global.use_det": False,
        "Global.use_cls": False,
        "Global.use_rec": True,
        "Rec.lang_type": "en",
    })
    _ = rapidocr_predict(samples[0].image)

    for sample in samples:
        started = time.perf_counter()
        prediction = rapidocr_predict(sample.image)
        elapsed = time.perf_counter() - started
        rapidocr_rows.append(score_prediction("RapidOCR", sample, prediction, elapsed))

    print(f"Scored {len(rapidocr_rows)} RapidOCR predictions.")
else:
    print("RapidOCR arm disabled with RUN_RAPIDOCR=False.")


## 6. LLM-adjacent recognizer: TrOCR

TrOCR represents an image as patches, encodes them with a vision Transformer, and autoregressively generates wordpiece tokens with a text Transformer. That language-generating decoder is why it is useful as an LLM-adjacent comparison.

The first run downloads the model checkpoint. Model loading time is shown separately and excluded from per-image latency. On Apple Silicon the notebook prefers MPS; otherwise it uses CUDA when available, then CPU.


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel


def choose_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def synchronize(device: torch.device) -> None:
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()


device = choose_device()
trocr_rows: list[dict] = []
processor = None
trocr_model = None

if RUN_TROCR:
    load_started = time.perf_counter()
    processor = TrOCRProcessor.from_pretrained(MODEL_ID, use_fast=False)
    trocr_model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to(device).eval()
    load_elapsed = time.perf_counter() - load_started
    parameter_count = sum(parameter.numel() for parameter in trocr_model.parameters())
    print(f"Loaded {MODEL_ID} on {device} in {load_elapsed:.2f} s")
    print(f"Parameters: {parameter_count / 1_000_000:.1f} million")
else:
    print("TrOCR arm disabled with RUN_TROCR=False.")


In [ ]:
def trocr_predict(image: Image.Image) -> str:
    assert processor is not None and trocr_model is not None
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
    with torch.inference_mode():
        generated_ids = trocr_model.generate(pixel_values, max_new_tokens=64)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]


if RUN_TROCR:
    _ = trocr_predict(samples[0].image)
    synchronize(device)

    for sample in samples:
        synchronize(device)
        started = time.perf_counter()
        prediction = trocr_predict(sample.image)
        synchronize(device)
        elapsed = time.perf_counter() - started
        trocr_rows.append(score_prediction("TrOCR Small", sample, prediction, elapsed))

    print(f"Scored {len(trocr_rows)} TrOCR predictions.")


## 7. Compare accuracy and latency

The summary uses macro averages: every image contributes equally. If one arm was skipped, the tables still show the available arm, but no two-model conclusion should be drawn.


In [ ]:
results = pd.DataFrame(rapidocr_rows + trocr_rows)
if results.empty:
    raise RuntimeError("No OCR arm ran. Enable a model and satisfy its prerequisites.")

overall = (
    results.groupby("model", as_index=False)
    .agg(
        samples=("sample_id", "count"),
        exact_match_rate=("exact_match", "mean"),
        strict_cer=("cer", "mean"),
        casefold_cer=("cer_casefold", "mean"),
        casefold_wer=("wer_casefold", "mean"),
        median_latency_ms=("latency_ms", "median"),
    )
    .sort_values("casefold_cer")
)
overall.style.format({
    "exact_match_rate": "{:.1%}",
    "strict_cer": "{:.3f}",
    "casefold_cer": "{:.3f}",
    "casefold_wer": "{:.3f}",
    "median_latency_ms": "{:.1f}",
})


In [ ]:
by_condition = (
    results.groupby(["condition", "model"], as_index=False)
    .agg(strict_cer=("cer", "mean"), casefold_cer=("cer_casefold", "mean"))
)
cer_pivot = by_condition.pivot(index="condition", columns="model", values="casefold_cer")
cer_pivot = cer_pivot.reindex(CONDITIONS)
cer_pivot.style.format("{:.3f}").background_gradient(cmap="YlOrRd", axis=None)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

cer_pivot.plot(kind="bar", ax=axes[0], color=["#28536B", "#D17A22"][: len(cer_pivot.columns)])
axes[0].set_title("Case-folded character error rate by condition")
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean case-folded CER (lower is better)")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y", alpha=0.25)

latency_order = overall.sort_values("median_latency_ms")
axes[1].bar(latency_order["model"], latency_order["median_latency_ms"], color="#4C956C")
axes[1].set_title("Warm median recognition latency")
axes[1].set_ylabel("Milliseconds per line (lower is better)")
axes[1].grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


## 8. Inspect the errors, not just the score

A generative decoder may produce plausible text that is not visually present. For OCR, a fluent substitution can be more dangerous than an obvious garble. Inspect every non-exact prediction before interpreting the aggregate score.


In [ ]:
errors = results.loc[results["cer"] > 0, [
    "model", "condition", "truth", "prediction", "cer", "cer_casefold", "wer_casefold"
]].sort_values(["condition", "truth", "model"])

if errors.empty:
    print("Every prediction was exact on this run. Increase degradation to find failure modes.")
else:
    display(errors.style.format({
        "cer": "{:.3f}", "cer_casefold": "{:.3f}", "wer_casefold": "{:.3f}"
    }))


## 9. Interpretation checklist

Use the executed results to answer these rather than assuming one architecture wins:

1. Which model has lower case-folded CER/WER, and which better preserves exact casing?
2. Which degradation creates the largest error increase for each model?
3. Are TrOCR's errors plausible substitutions or obvious failures?
4. Is the accuracy change worth the latency, memory, model-download, and deployment cost?
5. Would the conclusion survive real scans, multiple fonts, languages, handwriting, and full-page layout?

### Architectural trade-offs

| Dimension | RapidOCR | TrOCR Small Printed |
|---|---|---|
| Core recognizer | PP-OCR ONNX model with CTC decoding | Vision Transformer + autoregressive text Transformer |
| Input used here | Single line (detection/classification disabled) | Single line (checkpoint's intended use) |
| Layout/detection | Available in the full engine | Not provided by this checkpoint |
| Language behavior | CTC sequence decoding with an OCR character dictionary | Generates wordpiece tokens conditioned on image and prior tokens |
| Deployment | Python wheel with bundled models + ONNX Runtime | PyTorch/Transformers runtime + cached model weights |
| Key risk | Character confusion and brittle visual errors | Fluent-looking substitutions or normalization |

**Practical default:** start with the smaller dedicated OCR pipeline when its languages and layouts fit, then justify a Transformer/VLM with a representative evaluation set. For high-stakes extraction, retain image-to-text provenance and validate critical fields regardless of model family.


## 10. Try one of your own line crops

Set `IMAGE_PATH` and `GROUND_TRUTH`, then run the cell. Crop to one text line for a fair TrOCR comparison.


In [ ]:
IMAGE_PATH: str | None = None
GROUND_TRUTH = ""

if IMAGE_PATH is not None:
    own_image = Image.open(IMAGE_PATH).convert("RGB")
    display(own_image)

    if rapidocr_engine is not None:
        own_rapidocr = normalize_text(rapidocr_predict(own_image))
        print(f"RapidOCR: {own_rapidocr!r}")

    if trocr_model is not None:
        own_trocr = normalize_text(trocr_predict(own_image))
        print(f"TrOCR:    {own_trocr!r}")

    if GROUND_TRUTH:
        print("Add this sample to the benchmark corpus for a scored comparison.")
else:
    print("Set IMAGE_PATH to evaluate a cropped text-line image.")


## Sources

- [Issue #80: to practice Optical Character Recognition](https://github.com/pomodorozhong/personal-research/issues/80)
- [RapidOCR installation guide](https://rapidai.github.io/RapidOCRDocs/main/en/install_usage/rapidocr/install/) and [usage guide](https://rapidai.github.io/RapidOCRDocs/main/install_usage/rapidocr/usage/)
- Li et al., [TrOCR: Transformer-based Optical Character Recognition with Pre-trained Models](https://arxiv.org/abs/2109.10282)
- [Microsoft TrOCR Small Printed model card](https://huggingface.co/microsoft/trocr-small-printed)
- [Hugging Face vision encoder-decoder documentation](https://huggingface.co/docs/transformers/model_doc/vision-encoder-decoder)
